# Lesson 04: MDLM — Masked Discrete Language Model

MDLM simplifies discrete diffusion to just masking and unmasking. We implement it and compare with D3PM.

**Paper reference:** Sahoo et al., 2024 — Simple and Effective Masked Diffusion Language Models.

In [ ]:
%pip install torch numpy matplotlib datasets tqdm --quiet

In [ ]:
import sys
sys.path.insert(0, '../../..')

import math
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from shared.utils.seed import set_seed
from shared.utils.device import get_device
from shared.datasets.text import SimpleTokenizer, TextDataset

set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. The Masking Schedule

MDLM's forward process is simply: mask each token with probability $\gamma(t)$ at time $t \in [0, 1]$.

In [ ]:
# Visualize the masking schedules
t = torch.linspace(0, 1, 200)

gamma_linear = t  # linear schedule
gamma_cosine = 1.0 - torch.cos(t * math.pi / 2)  # cosine schedule

plt.figure(figsize=(8, 4))
plt.plot(t, gamma_linear, label='Linear: gamma(t) = t')
plt.plot(t, gamma_cosine, label='Cosine: gamma(t) = 1 - cos(t*pi/2)')
plt.xlabel('Time t')
plt.ylabel('Masking Rate gamma(t)')
plt.title('MDLM Masking Schedules')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Cosine schedule masks slowly at first (preserving structure longer),")
print("then rapidly masks towards the end.")

## 2. Visualize the Masking Forward Process

In [ ]:
from src.mdlm import MDLM, MDLMDenoiser

# Prepare data
texts = [
    "the cat sat on the mat",
    "a dog ran in the park",
    "the sun is bright today",
    "birds fly in the sky",
    "fish swim in the sea",
] * 200

tokenizer = SimpleTokenizer(texts, level="char")
dataset = TextDataset(texts, tokenizer, seq_len=32)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Build a temporary MDLM just to visualize masking
dummy_denoiser = MDLMDenoiser(tokenizer.vocab_size, d_model=32, n_heads=2, n_layers=1, max_seq_len=32).to(device)
mdlm_viz = MDLM(dummy_denoiser, tokenizer.vocab_size, tokenizer.mask_id, device=device)

# Show masking at different times
sentence = "the cat sat on the mat"
x_0 = torch.tensor([tokenizer.encode(sentence)[:32]], device=device)
print(f"t=0.00: {sentence}")

for t_val in [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]:
    t = torch.tensor([t_val], device=device)
    x_t = mdlm_viz.mask_tokens(x_0, t)
    decoded = tokenizer.decode(x_t[0].cpu().tolist())
    n_masked = (x_t == tokenizer.mask_id).sum().item()
    print(f"t={t_val:.2f}: {decoded}  ({n_masked} masked)")

## 3. Train MDLM

In [ ]:
# Build the model
denoiser = MDLMDenoiser(
    vocab_size=tokenizer.vocab_size,
    d_model=64,
    n_heads=4,
    n_layers=3,
    max_seq_len=32,
    dropout=0.1,
).to(device)

mdlm = MDLM(
    denoiser=denoiser,
    vocab_size=tokenizer.vocab_size,
    mask_token_id=tokenizer.mask_id,
    num_timesteps=100,
    schedule_type="cosine",
    device=device,
)

n_params = sum(p.numel() for p in denoiser.parameters())
print(f"MDLM parameters: {n_params:,}")

In [ ]:
optimizer = torch.optim.Adam(denoiser.parameters(), lr=3e-4)
num_epochs = 20

losses = []
for epoch in range(num_epochs):
    epoch_loss = 0.0
    n_batches = 0
    
    for batch in dataloader:
        batch = batch.to(device)
        
        loss = mdlm.train_loss(batch)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(denoiser.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MDLM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Generate Samples

In [ ]:
# Generate samples
print("MDLM samples (temperature=0.8):")
samples = mdlm.sample(batch_size=5, seq_len=32, temperature=0.8, verbose=True)

print("\nGenerated text:")
for i, sample in enumerate(samples):
    text = tokenizer.decode(sample.cpu().tolist())
    print(f"  [{i+1}] '{text}'")

In [ ]:
# Effect of number of sampling steps
for n_steps in [10, 50, 100, 200]:
    mdlm.num_timesteps = n_steps
    samples = mdlm.sample(batch_size=3, seq_len=32, temperature=0.8)
    print(f"\nSampling steps = {n_steps}:")
    for sample in samples:
        text = tokenizer.decode(sample.cpu().tolist())
        print(f"  '{text}'")

# Reset
mdlm.num_timesteps = 100

## 5. MDLM vs D3PM Comparison

The key simplification: MDLM doesn't need transition matrices or posterior computation. It's just masking and predicting.

In [ ]:
print("Comparison summary:")
print()
print("D3PM:")
print("  - General transition matrices Q_t")
print("  - Supports uniform and absorbing corruption")
print("  - VLB loss with posterior computation")
print("  - Reverse step requires Q_t and Q_bar_{t-1}")
print()
print("MDLM:")
print("  - Only masking (absorbing transitions)")
print("  - Simple cross-entropy loss on masked positions")
print("  - Reverse step: just predict and unmask")
print("  - Continuous-time formulation")
print("  - Generally simpler to implement and tune")

## Key Takeaways

1. MDLM simplifies discrete diffusion to mask-and-predict.
2. The cosine masking schedule preserves structure longer than linear.
3. Sampling is iterative unmasking, which is intuitive and effective.
4. The loss only trains on masked positions, focusing model capacity where it matters.

## What's Next

In Lesson 05, we explore **practical training and sampling details** — learning rate schedules, loss weighting, and comparing D3PM vs MDLM on the same data.